# Bronze Layer - Raw Data Ingestion

This notebook extracts raw data from all API endpoints with deduplication, validation, and quarantine mechanisms.

## 1. Setup and Logging Configuration

In [1]:
# Bronze Layer: Raw Data Ingestion from Blue Owls API
# This notebook extracts data from all API endpoints and saves as CSV with metadata columns

import sys
from pathlib import Path

work_dir = Path("/home/jovyan/work")
if str(work_dir) not in sys.path:
    sys.path.insert(0, str(work_dir))

import os
import json
import logging
from datetime import datetime
import pandas as pd
from api_client import APIClient

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# Configuration
API_BASE_URL = os.environ.get("API_BASE_URL", "http://mock-api:8000")
API_USERNAME = os.environ.get("API_USERNAME", "candidate")
API_PASSWORD = os.environ.get("API_PASSWORD", "blue-owls-2026")

# Paths
OUTPUT_DIR = Path("/home/jovyan/work/output/bronze")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MANIFEST_FILE = OUTPUT_DIR / ".manifest.json"
DATA_START_DATE = "2018-07-01"

logger.info(f"Starting ingestion to {OUTPUT_DIR}")

2026-03-30 11:53:17,015 - INFO - Starting ingestion to /home/jovyan/work/output/bronze


## 2. Manifest Utilities and Tracking

In [2]:
def load_manifest() -> dict:
    """Load the ingestion manifest to track completed pages."""
    if MANIFEST_FILE.exists():
        with open(MANIFEST_FILE, 'r') as f:
            return json.load(f)
    return {}

def save_manifest(manifest: dict):
    """Save the ingestion manifest."""
    with open(MANIFEST_FILE, 'w') as f:
        json.dump(manifest, f, indent=2, default=str)

def page_key(endpoint: str, page: int) -> str:
    """Generate a unique key for a page."""
    return f"{endpoint}:page_{page}"

def has_page_been_ingested(manifest: dict, endpoint: str, page: int) -> bool:
    """Check if a page has already been ingested."""
    key = page_key(endpoint, page)
    return key in manifest and manifest[key].get("status") == "completed"

# Load existing manifest
manifest = load_manifest()
logger.info(f"Loaded manifest: {len(manifest)} entries")
print(manifest)

2026-03-30 11:53:17,383 - INFO - Loaded manifest: 6 entries


{'orders_last_run': {'timestamp': '2026-03-30T11:40:48.780156', 'status': 'completed', 'records_ingested': 22, 'records_duplicated': 978, 'records_errored': 0, 'total_records': 1066, 'total_quarantined': 0}, 'order_items_last_run': {'timestamp': '2026-03-30T11:40:48.914305', 'status': 'completed', 'records_ingested': 1000, 'records_duplicated': 0, 'records_errored': 0, 'total_records': 4000, 'total_quarantined': 0}, 'customers_last_run': {'timestamp': '2026-03-30T11:40:49.011672', 'status': 'completed', 'records_ingested': 1000, 'records_duplicated': 0, 'records_errored': 0, 'total_records': 4000, 'total_quarantined': 0}, 'products_last_run': {'timestamp': '2026-03-30T11:40:49.082090', 'status': 'completed', 'records_ingested': 1000, 'records_duplicated': 0, 'records_errored': 0, 'total_records': 4000, 'total_quarantined': 0}, 'sellers_last_run': {'timestamp': '2026-03-30T11:40:55.193134', 'status': 'completed', 'records_ingested': 1000, 'records_duplicated': 0, 'records_errored': 0, '

## 3. Ingestion Functions with Validation and Deduplication

In [3]:
def validate_record(record: dict, endpoint: str) -> tuple[bool, str]:
    """
    Validate record payload.
    
    Args:
        record: Record to validate
        endpoint: Endpoint name for context
        
    Returns:
        Tuple of (is_valid, error_message)
    """
    if not isinstance(record, dict):
        return False, f"Record is not a dict: {type(record)}"
    
    if not record:  # Empty record
        return False, "Record is empty"
    
    # Check for null/None values in critical fields (at least one non-null field expected)
    has_content = any(v is not None and v != "" for v in record.values())
    if not has_content:
        return False, "Record contains only null/empty values"
    
    return True, ""

def hash_record_for_dedup(record: dict) -> int:
    """
    Create a hash of record for O(1) duplicate detection.
    Hash all columns except metadata (_* columns).
    """
    record_without_meta = {k: v for k, v in sorted(record.items()) if not k.startswith('_')}
    return hash(frozenset(record_without_meta.items()))

def ingest_endpoint(
    client: APIClient,
    endpoint: str,
    source_name: str,
    output_file: Path,
    date_supported: bool = False,
    page_size: int = 1000
):
    """
    Ingest all data from an endpoint with deduplication, validation, and error handling.
    
    Args:
        client: APIClient instance
        endpoint: API endpoint path
        source_name: Human-readable source name
        output_file: Output CSV file path
        date_supported: Whether endpoint supports date_from/date_to parameters
        page_size: Records per page
    """
    manifest = load_manifest()
    all_records = []
    seen_hashes = set()  # O(1) deduplication using hashes
    quarantine_records = []  # Store malformed records
    
    # Load existing data if file exists (for append-only bronze layer)
    if output_file.exists():
        existing_df = pd.read_csv(output_file)
        all_records = existing_df.to_dict('records')
        # Pre-populate seen_hashes with existing records for O(1) duplicate detection
        for rec in all_records:
            seen_hashes.add(hash_record_for_dedup(rec))
        logger.info(f"Loaded {len(all_records)} existing records from {output_file}")
    
    params = {}
    if date_supported:
        params["date_from"] = DATA_START_DATE
    
    ingested_count = 0
    skipped_count = 0
    error_count = 0
    
    try:
        logger.info(f"Starting ingestion from {source_name}")
        
        for record in client.fetch_paginated(endpoint, params=params, page_size=page_size):
            # Validate record payload
            is_valid, error_msg = validate_record(record, endpoint)
            if not is_valid:
                logger.warning(f"Malformed record: {error_msg}")
                record['_ingested_at'] = datetime.utcnow().isoformat()
                record['_source_endpoint'] = endpoint
                record['_error'] = error_msg
                quarantine_records.append(record)
                error_count += 1
                continue
            
            # Add metadata columns
            record['_ingested_at'] = datetime.utcnow().isoformat()
            record['_source_endpoint'] = endpoint
            
            # O(1) duplicate check using hash
            record_hash = hash_record_for_dedup(record)
            if record_hash not in seen_hashes:
                all_records.append(record)
                seen_hashes.add(record_hash)
                ingested_count += 1
            else:
                skipped_count += 1
        
        # Save valid records to CSV
        if all_records:
            df = pd.DataFrame(all_records)
            df.to_csv(output_file, index=False)
            logger.info(
                f"Saved {len(all_records)} total records to {output_file.name} "
                f"(ingested: {ingested_count}, duplicates: {skipped_count}, errors: {error_count})"
            )
        
        # Save quarantined records if any errors occurred
        if quarantine_records:
            quarantine_file = output_file.parent / f"{source_name}_quarantine.csv"
            df_quarantine = pd.DataFrame(quarantine_records)
            df_quarantine.to_csv(quarantine_file, index=False)
            logger.warning(f"Saved {len(quarantine_records)} quarantined records to {quarantine_file.name}")
        
        # Update manifest with complete tracking
        manifest[f"{source_name}_last_run"] = {
            "timestamp": datetime.utcnow().isoformat(),
            "status": "completed",
            "records_ingested": ingested_count,
            "records_duplicated": skipped_count,
            "records_errored": error_count,
            "total_records": len(all_records),
            "total_quarantined": len(quarantine_records),
        }
        save_manifest(manifest)
        
    except Exception as e:
        logger.error(f"Error ingesting {source_name}: {e}")
        # Mark endpoint as failed in manifest for diagnostics
        manifest[f"{source_name}_last_run"] = {
            "timestamp": datetime.utcnow().isoformat(),
            "status": "failed",
            "error": str(e),
            "records_ingested": ingested_count,
        }
        save_manifest(manifest)
        raise

logger.info("Ingestion functions defined")

2026-03-30 11:53:17,793 - INFO - Ingestion functions defined


## 4. Initialize API Client and Endpoints

In [4]:
# Initialize API client
client = APIClient(
    base_url=API_BASE_URL,
    username=API_USERNAME,
    password=API_PASSWORD,
)

# Define endpoints to ingest
endpoints = [
    {
        "path": "api/v1/data/orders",
        "name": "orders",
        "date_supported": True,
    },
    {
        "path": "api/v1/data/order_items",
        "name": "order_items",
        "date_supported": True,
    },
    {
        "path": "api/v1/data/customers",
        "name": "customers",
        "date_supported": False,
    },
    {
        "path": "api/v1/data/products",
        "name": "products",
        "date_supported": False,
    },
    {
        "path": "api/v1/data/sellers",
        "name": "sellers",
        "date_supported": False,
    },
    {
        "path": "api/v1/data/payments",
        "name": "payments",
        "date_supported": True,
    },
]

logger.info(f"Starting bronze layer ingestion for {len(endpoints)} endpoints")
print(f"Output directory: {OUTPUT_DIR}")

2026-03-30 11:53:18,170 - INFO - Starting bronze layer ingestion for 6 endpoints


Output directory: /home/jovyan/work/output/bronze


## 5. Execute Ingestion for All Endpoints

In [5]:
# Ingest all endpoints
ingestion_results = {}

for endpoint_config in endpoints:
    endpoint_path = endpoint_config["path"]
    endpoint_name = endpoint_config["name"]
    output_file = OUTPUT_DIR / f"{endpoint_name}.csv"
    
    try:
        ingest_endpoint(
            client=client,
            endpoint=endpoint_path,
            source_name=endpoint_name,
            output_file=output_file,
            date_supported=endpoint_config.get("date_supported", False),
        )
        ingestion_results[endpoint_name] = "SUCCESS"
    except Exception as e:
        ingestion_results[endpoint_name] = f"ERROR: {str(e)}"
        logger.error(f"Failed to ingest {endpoint_name}: {e}")

print("\n" + "="*60)
print("INGESTION SUMMARY")
print(""*60)
for endpoint, status in ingestion_results.items():
    print(f"{endpoint:20s}: {status}")
print(""*60)

2026-03-30 11:53:18,580 - INFO - Loaded 1066 existing records from /home/jovyan/work/output/bronze/orders.csv
2026-03-30 11:53:18,581 - INFO - Starting ingestion from orders
2026-03-30 11:53:18,608 - INFO - Token obtained, expires at 2026-03-30 12:06:18.608268


2026-03-30 11:53:18,708 - INFO - Saved 1088 total records to orders.csv (ingested: 22, duplicates: 978, errors: 0)
2026-03-30 11:53:18,760 - INFO - Loaded 4000 existing records from /home/jovyan/work/output/bronze/order_items.csv
2026-03-30 11:53:18,761 - INFO - Starting ingestion from order_items
2026-03-30 11:53:18,867 - INFO - Saved 5000 total records to order_items.csv (ingested: 1000, duplicates: 0, errors: 0)
2026-03-30 11:53:18,921 - INFO - Loaded 4000 existing records from /home/jovyan/work/output/bronze/customers.csv
2026-03-30 11:53:18,922 - INFO - Starting ingestion from customers
2026-03-30 11:53:18,984 - INFO - Saved 5000 total records to customers.csv (ingested: 1000, duplicates: 0, errors: 0)
2026-03-30 11:53:19,038 - INFO - Loaded 4000 existing records from /home/jovyan/work/output/bronze/products.csv
2026-03-30 11:53:19,039 - INFO - Starting ingestion from products
2026-03-30 11:53:19,125 - INFO - Saved 5000 total records to products.csv (ingested: 1000, duplicates: 0,


INGESTION SUMMARY

orders              : SUCCESS
order_items         : SUCCESS
customers           : SUCCESS
products            : SUCCESS
sellers             : SUCCESS
payments            : SUCCESS



## 6. Bronze Layer File Verification

In [6]:
# Verify bronze layer files
print("\nBRONZE LAYER FILES:")
print(""*60)
bronze_files = list(OUTPUT_DIR.glob("*.csv"))
for file in sorted(bronze_files):
    df = pd.read_csv(file)
    print(f"{file.name:30s}: {len(df):8d} rows, {len(df.columns):3d} columns")
print(""*60)


BRONZE LAYER FILES:

customers.csv                 :     5000 rows,   7 columns
order_items.csv               :     5000 rows,   9 columns
orders.csv                    :     1088 rows,  10 columns
payments.csv                  :     5000 rows,   7 columns
products.csv                  :     5000 rows,  11 columns
sellers.csv                   :     5000 rows,   6 columns

